# ⚾ MLB Pitch Bot – Model Prototype v2.1

This notebook explores the highly-accurate **Pitch Family** model enhanced with **Handedness** and **Base State**.

## 🧠 Modeling Philosophy
1. **Family Target**: Predict Fastball, Breaking, Offspeed.
2. **Advanced Context**: Added Pitcher/Batter Handedness and Men On Base state.
3. **No Leaky Features**: Avoiding velocity/spin.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt

# Project root for src imports
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.database import query_all_pitches
from src.dataset_generator import add_features
from src.constants import DATABASE_PATH
from src.features import _classify_pitch_family

## 1. Data Ingestion
Load raw pitches with new features.

In [ ]:
df_raw = query_all_pitches()
print(f"Loaded {len(df_raw)} pitches.")
df = add_features(df_raw)
df.head()

## 2. Advanced Feature Prep

In [ ]:
df = df.dropna(subset=["pitch_type"]).copy()
df["pitch_family"] = df["pitch_type"].apply(_classify_pitch_family)

y_encoder = LabelEncoder()
y = y_encoder.fit_transform(df["pitch_family"])
class_names = y_encoder.classes_

# Advanced Categorical Features
p_le, b_le, m_le = LabelEncoder(), LabelEncoder(), LabelEncoder()
df["p_hand_enc"] = p_le.fit_transform(df["pitcher_hand"].fillna("R"))
df["b_side_enc"] = b_le.fit_transform(df["batter_side"].fillna("R"))
df["mob_enc"] = m_le.fit_transform(df["men_on_base"].fillna("Empty"))
df["prev_f_enc"] = LabelEncoder().fit_transform(df["prev_pitch_type_in_ab"].apply(_classify_pitch_family))

feat_cols = ["p_hand_enc", "b_side_enc", "mob_enc", "prev_f_enc", "inning", "balls", "strikes", "outs", "is_leverage"]
tendency_cols = [c for c in df.columns if "tendency_" in c and "_pct" in c]
X_cols = feat_cols + tendency_cols
X = df[X_cols].fillna(0)

print(f"Features used: {X_cols}")

## 3. Training & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    eval_metric="mlogloss"
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names))

## 4. Feature Importance

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=X_cols).sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette="magma")
plt.title("Impact of Handedness & Base State on Pitch Choice")
plt.show()